# Step 2: Create Features (ATR)
This notebook calculates the Average True Range (ATR) for every stock in our tradable universe.

**Workflow:**
1. Load the raw OHLCV data (`top_5000_yf_data.pkl`)
2. Load the universe mask (`stores/universe_5m.parquet`) created by notebook 1
3. Calculate ATR using the vectorized function from `technical_indicators.py`
4. Mask the ATR so only in-universe stocks have values (out-of-universe → NaN)
5. Save to `stores/features.parquet` for downstream use

In [1]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))
from technical_indicators import calculate_atr_vectorized

import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "stores"
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
# Load raw OHLCV data
print('Loading top_5000_yf_data.pkl...')
pv = pd.read_pickle('top_5000_yf_data.pkl')
print(f'Data loaded. Shape: {pv.shape}')
print(f'Date range: {pv.index[0].date()} → {pv.index[-1].date()}')
print(f'Tickers: {len(pv["Close"].columns)}')

Loading top_5000_yf_data.pkl...
Data loaded. Shape: (4092, 29994)
Date range: 2010-01-04 → 2026-04-10
Tickers: 4999


In [3]:
# Load the tradable universe mask from notebook 1
universe_path = os.path.join(DATA_DIR, 'universe_5m.parquet')
df_universe = pd.read_parquet(universe_path)
print(f'Universe mask loaded. Shape: {df_universe.shape}')
print(f'Avg tradable stocks per day: {df_universe.sum(axis=1).mean():.0f}')

Universe mask loaded. Shape: (4092, 4999)
Avg tradable stocks per day: 619


In [4]:
# Calculate ATR using vectorized function (runs in seconds)
print('Calculating ATR...')
indicators_dict = calculate_atr_vectorized(pv)
atr_all = indicators_dict['average_true_range']
print(f'Raw ATR shape: {atr_all.shape}')

Calculating ATR...
Raw ATR shape: (4092, 4999)


In [5]:
# Align ATR with the universe mask
# Only keep ATR values for stocks that are in the tradable universe on each day
# Out-of-universe stocks get NaN
common_dates = atr_all.index.intersection(df_universe.index)
common_tickers = atr_all.columns.intersection(df_universe.columns)

atr_aligned = atr_all.loc[common_dates, common_tickers]
universe_aligned = df_universe.loc[common_dates, common_tickers]

# Apply the mask: where universe == 0, set ATR to NaN
atr_masked = atr_aligned.where(universe_aligned == 1)

print(f'Masked ATR shape: {atr_masked.shape}')
print(f'Avg stocks with valid ATR per day: {atr_masked.notna().sum(axis=1).mean():.0f}')

Masked ATR shape: (4092, 4999)
Avg stocks with valid ATR per day: 619


In [6]:
# Downcast to float32 to save memory
atr_masked = atr_masked.astype('float32')

# Preview
print('\nSample ATR values (last 5 days, first 10 in-universe stocks):')
last_day = atr_masked.iloc[-1]
valid_tickers = last_day.dropna().index[:10].tolist()
atr_masked[valid_tickers].tail()


Sample ATR values (last 5 days, first 10 in-universe stocks):


Ticker,AA,AAL,AAON,AAP,ABAT,ABCL,ABSI,ABUS,ABVX,ABX
Date,,,,,,,,,,
2026-04-06,3.748857,NaN,4.508572,2.361429,0.188857,0.169357,0.267857,0.210714,5.998500,0.488000
2026-04-07,3.714856,0.480714,4.377143,2.232857,0.188500,0.175429,0.241071,0.210714,6.103500,0.513714
2026-04-08,3.904856,0.548571,4.739287,2.235001,0.192786,0.181857,0.237857,0.217857,6.097072,0.507286
2026-04-09,3.584141,0.540714,4.856430,2.320715,NaN,0.173643,0.223929,0.212143,6.277072,0.501571
2026-04-10,3.457713,0.523571,4.808572,2.420001,0.183143,0.173286,0.215357,0.209286,6.207072,0.490857


In [7]:
# Save as MultiIndex DataFrame under key 'average_true_range'
# This format is consistent with adding more features later
features_df = pd.concat({'average_true_range': atr_masked}, axis=1)
features_df = features_df.astype('float32')

output_path = os.path.join(DATA_DIR, 'features.parquet')
features_df.to_parquet(output_path, compression='zstd', engine='pyarrow')
print(f'Saved to {output_path}')
print(f'Final features shape: {features_df.shape}')
print(f'Memory usage: {features_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

Saved to stores\features.parquet
Final features shape: (4092, 4999)
Memory usage: 81.9 MB
